# Zero Trust Architecture

## Learning objectives

By the end of this notebook, you should be able to:

1. Explain what zero trust means.
2. Identify the main components of a zero trust architecture.
3. Describe CISA’s five zero trust pillars.
4. Trace an access request from start to finish.
5. Apply least-privilege rules to a practical example.
6. Plan and test a basic zero trust implementation.

> The Python examples in this notebook demonstrate access-control logic for learning. They are not production security controls.

# 1. What Is Zero Trust?

Zero trust is a cybersecurity approach in which access to a resource is evaluated using explicit policies. A user or device does not receive access simply because it is connected to an organization’s internal network.

The central question is:

**Should this identity, using this device, be allowed to perform this action on this resource under the current conditions?**

A resource could be a web application, database, API, file, cloud workload, or administrative system.

## Three practical principles

### 1. Verify explicitly

Check relevant information before granting access. This may include the user's identity, authentication method, device condition, permissions, and the sensitivity of the resource.

### 2. Limit privileges

Grant only the access needed for the task. Permission to use an application does not automatically include permission to administer its server or export all its data.

### 3. Reassess access

Monitor relevant changes during a session. If an account is disabled, a device becomes noncompliant, or activity becomes suspicious, the organization may require another check or revoke access.

**Important:** Zero trust does not mean “trust nobody.” It means that trust is not assumed solely from network location. Access is evaluated for a particular request.

**Source:** NIST Special Publication 800-207, *Zero Trust Architecture*.

# 2. Main Components of a Zero Trust Architecture

A zero trust architecture separates the decision to allow access from the component that enforces that decision.

| Component | Purpose | Example |
|---|---|---|
| Subject | The person or service requesting access | An HR employee |
| Resource | The protected application or data | Payroll application |
| Policy engine (PE) | Decides whether to allow, deny, or revoke access | Evaluates role, MFA, and device status |
| Policy administrator (PA) | Establishes or ends access based on the decision | Tells a gateway to permit a session |
| Policy enforcement point (PEP) | Enforces the decision in the access path | Application gateway |
| Information sources | Supply signals needed for decisions | Identity provider and device management system |

## Simplified access flow

1. A subject requests a resource.
2. The policy enforcement point receives or intercepts the request.
3. The policy engine evaluates the request using policy and available signals.
4. The policy administrator communicates the decision.
5. The enforcement point allows or denies the connection.
6. Activity is monitored, and access can be reassessed.

The enforcement point must protect the actual route to the resource. If a user can reach the resource through another unprotected path, the policy can be bypassed.

**Sources:** NIST SP 800-207 and the NIST Zero Trust Architecture Project Glossary.

# 3. CISA’s Five Zero Trust Pillars

CISA groups zero trust capabilities into five pillars.

| Pillar | Main question | Example controls |
|---|---|---|
| Identity | Who is requesting access? | Unique accounts, multifactor authentication, account lifecycle management |
| Devices | Is the device known and in an acceptable condition? | Inventory, patching, encryption, endpoint protection |
| Networks | Which network connections are permitted? | Segmentation, encrypted traffic, resource-specific gateways |
| Applications and workloads | Which applications, APIs, and services can be used? | Application authentication, API authorization, service identities |
| Data | What information can be viewed, changed, downloaded, or shared? | Classification, encryption, access controls, audit logs |

CISA also identifies capabilities that operate across the pillars:

- Visibility and analytics
- Automation and orchestration
- Governance

For example, confirming a user’s identity is useful, but it does not tell the organization whether the device is compromised or whether a bulk data export is authorized.

**Source:** CISA, *Zero Trust Maturity Model*, Version 2.0.

# 4. End-to-End Example: Accessing a Payroll Application

Assume Anita works in the HR department and wants to view an employee’s payroll record from her company laptop.

## Step 1: Identify the resource

The organization classifies payroll information as sensitive. It documents the payroll application, its database, authorized users, and ways the application can be reached.

## Step 2: Request access

Anita opens the payroll application. Her request reaches a policy enforcement point, such as an application gateway.

## Step 3: Verify identity

The identity provider verifies Anita’s account and checks whether she has completed the required multifactor authentication.

## Step 4: Check the device

The policy system checks whether Anita’s laptop is managed and meets required security conditions.

## Step 5: Check authorization

The policy engine determines whether Anita’s HR role permits the specific action she requested: viewing a payroll record.

## Step 6: Enforce the decision

If the request satisfies policy, the policy administrator instructs the enforcement point to permit a scoped session to the payroll application.

## Step 7: Protect the data

Anita may be allowed to view a record without being allowed to export the entire payroll database. Bulk export requires a separate authorization decision.

## Step 8: Monitor and reassess

The organization records relevant authentication and access events. If Anita’s account is disabled or the device becomes noncompliant, the system can reassess or revoke access according to policy.

**Key idea:** One successful sign-in does not give Anita unlimited access to every system or every action.

# 5. Example Access Policy

A human-readable payroll policy might say:

> Allow members of the HR payroll group to view payroll records through the payroll application when they have completed MFA and are using a compliant managed device. Deny direct database access. Require separate authorization for bulk exports.

This creates several distinct decisions:

| Request | Decision under this example policy |
|---|---|
| HR employee views an authorized payroll record from a compliant device | Allow |
| Same employee exports the entire database | Deny unless separately authorized |
| Contractor views payroll records | Deny |
| HR employee uses an unmanaged device | Deny |
| HR employee connects directly to the payroll database | Deny |

This is an example of **least privilege**: permission is limited by identity, resource, action, and conditions.

In [1]:
from dataclasses import dataclass

@dataclass(frozen=True)
class AccessRequest:
    user: str
    role: str
    resource: str
    action: str
    mfa: bool
    managed_device: bool
    device_compliant: bool


def evaluate(req: AccessRequest) -> tuple[bool, str]:
    if req.resource != "payroll_app":
        return False, "Resource is outside this policy"

    if not req.mfa:
        return False, "MFA required"

    if not (req.managed_device and req.device_compliant):
        return False, "Compliant managed device required"

    if req.role != "HR":
        return False, "HR role required"

    if req.action != "view_record":
        return False, "Action requires separate authorization"

    return True, "Scoped access granted"


requests = [
    AccessRequest(
        "Anita", "HR", "payroll_app", "view_record",
        True, True, True
    ),
    AccessRequest(
        "Anita", "HR", "payroll_app", "bulk_export",
        True, True, True
    ),
    AccessRequest(
        "Attacker", "HR", "payroll_app", "view_record",
        False, False, False
    ),
    AccessRequest(
        "Contractor", "Contractor", "payroll_app", "view_record",
        True, True, True
    ),
]

for req in requests:
    allowed, reason = evaluate(req)
    decision = "ALLOW" if allowed else "DENY"
    print(f"{req.user:10} {req.action:12} -> {decision:5} | {reason}")

Anita      view_record  -> ALLOW | Scoped access granted
Anita      bulk_export  -> DENY  | Action requires separate authorization
Attacker   view_record  -> DENY  | MFA required
Contractor view_record  -> DENY  | HR role required


## Understanding the Output

- Anita’s request to view a record is allowed because it satisfies every condition in this simplified policy.
- Anita’s bulk export is denied because viewing and exporting are separate actions.
- The attacker’s request is denied because the required authentication and device conditions are not satisfied.
- The contractor’s request is denied because the contractor does not have the required HR role.

**Security limitation:** In a real architecture, the requester must not be allowed to simply claim that `mfa=True` or `device_compliant=True`. Those values must come from trustworthy identity and device systems. The application must also enforce which individual records Anita is authorized to view.

# 6. How to Implement Zero Trust in an Organization

Zero trust is usually introduced in stages. Start with a valuable resource and a well-understood access workflow.

## Phase 1: Inventory

Identify users, service accounts, devices, applications, data, and existing access paths. Determine who owns each resource and who needs to use it.

**Output:** A resource inventory and access map.

## Phase 2: Prioritize a resource

Choose a sensitive application, such as payroll, that has a clear owner and manageable set of users.

**Output:** Documented access requirements and risks.

## Phase 3: Strengthen identity and device controls

Use unique accounts, MFA, controlled administrator privileges, device inventory, and appropriate device health checks.

**Output:** Trustworthy identity and device signals.

## Phase 4: Write access policies

Define which identities can perform which actions on which resources. Document contractor access, service accounts, exceptions, and emergency procedures.

**Output:** Reviewed policies that can be tested.

## Phase 5: Enforce the policies

Put enforcement at the relevant access paths. Check that there are no alternative routes around the enforcement point.

**Output:** Resource access controlled by policy.

## Phase 6: Protect data and collect telemetry

Classify sensitive data and control viewing, modification, export, and sharing. Collect logs needed to review access and investigate incidents.

**Output:** Data-level controls and useful audit records.

## Phase 7: Test and improve

Test legitimate use as well as unauthorized use. Review false denials, bypass paths, system failures, and changing business requirements.

**Output:** Test results and an updated implementation plan.

NIST SP 1800-35 provides practical example implementations that can help organizations plan integrations.

# 7. Threat Scenarios

| Threat | How zero trust controls may help | Limitation to consider |
|---|---|---|
| Stolen password | MFA and device checks can block access | Stolen sessions and sophisticated phishing remain concerns |
| Compromised laptop | Device monitoring and revocation can limit access | Detection may occur after the initial compromise |
| Malicious insider | Narrow permissions and data logs can limit and reveal misuse | Some harmful actions may fall within existing permissions |
| Lateral movement | Segmentation and resource-specific access reduce reach | Unprotected access paths can undermine the design |
| Compromised service account | Scoped service permissions limit accessible APIs | Secrets and tokens still require protection |

Zero trust reduces unnecessary access and improves opportunities to detect misuse. Its effectiveness depends on accurate policies, trustworthy signals, and enforcement on every relevant access path.

# 8. Review Questions

1. What is the difference between a policy engine and a policy enforcement point?
2. Why does MFA alone not create a complete zero trust architecture?
3. What could happen when a managed device becomes noncompliant during a session?
4. Why should viewing a payroll record and exporting the payroll database be separate permissions?
5. What is the risk of a direct network path that bypasses the enforcement point?

## Answers

1. The policy engine makes an access decision; the policy enforcement point applies it.
2. MFA checks authentication but does not establish device condition, resource authorization, or permitted actions.
3. The system can reassess the request and adjust or revoke access according to policy.
4. The actions have different business purposes and risks, so they need separate authorization.
5. A user or attacker may reach the resource without the intended policy being enforced.